In [32]:
!pip install pandas
!pip install pyvis


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import pandas as pd
from pyvis.network import Network

# Загрузка данных
df = pd.read_csv('FinFraud_Labelled.csv', sep='|', header=None)

df.columns = [
    'scenario', 'col1', 'col2', 'sender', 'receiver', 'amount',
    'type', 'status', 's_old_bal', 's_new_bal', 'r_old_bal', 'r_new_bal',
    'c12', 'c13', 'c14', 'c15', 'time_step', 'time_orig', 'c18', 'c19', 'c20', 'c21', 'c22', 'c23', 'c24'
]

# Удаляем пустые строки и приводим сценарии к тексту
df = df.dropna(subset=['scenario'])
df['scenario'] = df['scenario'].astype(str)

print("Данные успешно загружены. Всего строк:", len(df))

Данные успешно загружены. Всего строк: 54848


C:\Users\79818\AppData\Local\Temp\ipykernel_16060\3796824538.py:5: DtypeWarning: Columns (12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('FinFraud_Labelled.csv', sep='|', header=None)


In [53]:
#Убираем ArRC (оплата связи)  - захламляло граф для обычных пользователей.

df_normal_clean = df[
    (df['scenario'].str.startswith('N')) &
    (df['receiver'] != 'A0')
].head(100)

# Мошенники (их фильтровать не надо, нам важна любая их активность)
df_fraud_clean = df[df['scenario'].str.startswith('F')].head(200)

# Смешанный граф для сравнения
df_mixed_clean = pd.concat([
    df_normal_clean.head(100),
    df_fraud_clean.head(200)
])

print("Фильтрация завершена. Супер-узел A0 исключен из нормальных транзакций.")

Фильтрация завершена. Супер-узел A0 исключен из нормальных транзакций.


In [55]:
def build_graph(data, file_name):
    net = Network(height='800px', width='100%', bgcolor='#222222', font_color='white', directed=True, notebook=False)

    #Сначала считаем общий объем транзакций для каждого узла, чтобы определить его размер
    node_weights = {}
    for _, row in data.iterrows():
        s, r, a = str(row['sender']), str(row['receiver']), float(row['amount'])
        node_weights[s] = node_weights.get(s, 0) + a
        node_weights[r] = node_weights.get(r, 0) + a

    added_nodes = {}

    for _, row in data.iterrows():
        src, dst = str(row['sender']), str(row['receiver'])
        scenario, amt = str(row['scenario']), float(row['amount'])

        # Цвета
        if scenario == 'F_bot': color = '#00ffcc'   # Ботнет
        elif scenario.startswith('F'): color = '#ff4d4d' # Кража/Мул
        else: color = '#3498db'                     # Норма

        # Добавляем узлы с динамическим размером
        for node in [src, dst]:
            if node not in added_nodes:
                # Размер узла: берем корень от суммы, чтобы гиганты не поглотили экран
                base_size = 10
                weighted_size = base_size + (node_weights[node]**0.5 / 100)

                # Форма: если в названии есть RAcc - это треугольник (агент для вывода средств)
                shape = 'triangle' if 'RAcc' in node else 'dot'

                net.add_node(node, label=node, color=color, size=weighted_size, shape=shape)
                added_nodes[node] = True

        #Толщина ребра: используем значение суммы напрямую для атрибута value
        net.add_edge(src, dst, color=color, value=amt, title=f"Сумма: {amt:.2f}\nТип: {scenario}")

    net.barnes_hut(gravity=-10000, central_gravity=0.1, spring_length=300)
    net.write_html(file_name)
    print(f"✅ Улучшенный граф сохранен: {file_name}")

In [56]:

build_graph(df_normal_clean, 'Graph_1_Normal.html')

build_graph(df_fraud_clean, 'Graph_2_Fraud.html')

build_graph(df_mixed_clean, 'Graph_3_Mixed.html')

✅ Улучшенный граф сохранен: Graph_1_Normal.html
✅ Улучшенный граф сохранен: Graph_2_Fraud.html
✅ Улучшенный граф сохранен: Graph_3_Mixed.html
